# C-SALT Verification — Paper Table 1 (Śivopāsanā Dissections)
**v5 — 15 March 2026**

**Purpose:** Verify Sanskrit dissections in Table 1 of:
> *Vedic Cosmological Time Cycles and Geophysical Periodicities* (Chilakalapudi 2026, v8.6)

**Fixes in v5:**
- Step 1 now prints the **raw response** so we can see exactly what shape the API returns
- `extract_entries()`: no false warning on empty `{"data":[]}`, safe handling of `{"data": null}`
- GQL introspection: null-safe `resp.get('data') or {}` instead of `resp['data']`
- REST field matrix now includes `headword` (the field name may have changed from `headword_slp1`)
- GQL matrix tests `field: headword` and `field: headword_slp1` as separate entries
- Step 3 in Cell 2 adds `match_phrase` and `fuzzy` to REST query_type variants
- CONFIG auto-set from whatever Step 3/4 finds working

**Run on Google Colab — fresh IP**
**Cell order:** 1 → 2 (diagnostic) → 3 (functions+ping) → 4 (scan) → 5 (deep checks) → 6 (save)

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 1 — SETUP
# ════════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, urllib.request, urllib.parse, urllib.error
from datetime import datetime
from collections import defaultdict

for p in ['/content/drive/MyDrive/Panchakshari_Research',
          '/content/drive/MyDrive/PCH_Documents/Projetcs/Panchakshari Pravachanam',
          '/content/drive/MyDrive/Panchakshari Pravachanam',
          '/content/drive/MyDrive']:
    if os.path.exists(p):
        PROJECT = p; print(f'✓ Project: {p}'); break
else:
    PROJECT = '/content/drive/MyDrive/Panchakshari_Research'
    os.makedirs(PROJECT, exist_ok=True)

OUTPUT_DIR = f'{PROJECT}/csalt_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

CSALT_BASE = 'https://api.c-salt.uni-koeln.de/dicts'
DICTS   = {'mw': 'Monier-Williams', 'ap90': 'Apte 1890', 'pwg': 'Böhtlingk-Roth'}
GQL_URL = {k: f'{CSALT_BASE}/{k}/graphql' for k in DICTS}
H_REST  = {'Accept': 'application/json', 'User-Agent': 'PanchakshariVerify/5.0'}
H_GQL   = {'Content-Type': 'application/json', 'Accept': 'application/json',
            'User-Agent': 'PanchakshariVerify/5.0'}

CONFIG = {'field': None, 'qtype': None, 'use_gql': False,
          'gql_qtype': None, 'gql_field': None, 'api_alive': False, 'delay': 0.4}
STATE  = {'calls':0,'total_429':0,'total_403':0,'network_dead':False,'consec_errors':0}

WORDS = [
    ('viSva','viśva',1,'viśva = all/universe'),
    ('ISvara','īśvara',1,'īśvara = lord'),
    ('mahat','mahat',2,'mahat = great; Sāṃkhya first evolute'),
    ('deva','deva',2,'deva = god, shining one'),
    ('tri','tri',3,'tri = three (prefix)'),
    ('ambaka','ambaka',3,'KEY: ambaka = eye AND/OR of-mother?'),
    ('ambA','ambā',3,'ambā = mother (separate MW entry?)'),
    ('tripura','tripura',4,'tripura = three demon cities'),
    ('antaka','antaka',4,'antaka = ender, destroyer'),
    ('trikAla','trikāla',5,'trikāla = three times'),
    ('agni','agni',5,'agni = fire'),
    ('kAla','kāla',5,'KEY: kāla = time AND death?'),
    ('rudra','rudra',6,'rudra = Rudra'),
    ('nIla','nīla',7,'nīla = blue'),
    ('kaNTha','kaṇṭha',7,'kaṇṭha = throat'),
    ('mftyu','mṛtyu',8,'mṛtyu = death'),
    ('jaya','jaya',8,'jaya = victory'),
    ('yuj','yuj',8,'KEY: √yuj = to yoke/join'),
    ('sarva','sarva',9,'sarva = all'),
    ('sadA','sadā',10,'KEY: sadā = always (adverb)'),
    ('sat','sat',10,'sat = being/truth'),
    ('Siva','śiva',10,'śiva = auspicious'),
    ('SrImat','śrīmat',11,'śrīmat = glorious'),
    ('mahAdeva','mahādeva',11,'mahādeva = Great God compound'),
]
print(f'✓ {len(WORDS)} words × {len(DICTS)} dicts = {len(WORDS)*len(DICTS)} API calls')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 2 — FULL API DIAGNOSTIC (v5)
# KEY FIX: raw response dump, null-safe GQL, expanded field matrix
# ════════════════════════════════════════════════════════════════
print('=' * 65)
print('C-SALT API DIAGNOSTIC (v5)')
print('=' * 65)

def http_get_raw(url, timeout=15):
    """GET → (status, parsed_json_or_None, raw_text)"""
    try:
        req = urllib.request.Request(url, headers=H_REST)
        with urllib.request.urlopen(req, timeout=timeout) as r:
            raw = r.read().decode('utf-8')
            try: parsed = json.loads(raw)
            except: parsed = None
            return r.status, parsed, raw
    except urllib.error.HTTPError as e:
        return e.code, None, ''
    except Exception as e:
        return str(e)[:50], None, ''

def gql_post_raw(dict_id, query_str, timeout=20):
    """POST GQL → (status, response_dict_or_None)"""
    try:
        payload = json.dumps({'query': query_str}).encode()
        req = urllib.request.Request(GQL_URL[dict_id], data=payload, headers=H_GQL)
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return r.status, json.loads(r.read())
    except urllib.error.HTTPError as e:
        try: body = e.read().decode()[:200]
        except: body = ''
        return e.code, {'http_error': e.code, 'body': body}
    except Exception as e:
        return str(e)[:50], None

def flatten_entries(raw_parsed):
    """
    v5: Extract entries list from any response shape.
    Returns (entries_list, shape_description).
    Does NOT warn on empty lists — that's valid.
    """
    if raw_parsed is None:
        return [], 'null'
    if isinstance(raw_parsed, list):
        return raw_parsed, 'list'
    if isinstance(raw_parsed, dict):
        # Try common wrapper keys
        for key in ('results', 'entries', 'data', 'items', 'hits', 'docs', 'records'):
            val = raw_parsed.get(key)
            if val is None:
                continue
            if isinstance(val, list):
                return val, f'dict[{key}]=list'
            if isinstance(val, dict):
                # Nested: {hits: {hits: [...]}}
                for inner_key in ('entries', 'results', 'hits', 'docs'):
                    inner = val.get(inner_key)
                    if isinstance(inner, list):
                        # ES-style: each hit has _source
                        return [h.get('_source', h) for h in inner], f'dict[{key}][{inner_key}]'
        # Single entry dict?
        if 'headword_slp1' in raw_parsed or 'headword' in raw_parsed:
            return [raw_parsed], 'single_entry_dict'
        return [], f'unknown_dict_keys:{list(raw_parsed.keys())[:6]}'
    return [], f'unknown_type:{type(raw_parsed).__name__}'

# ── STEP 1: Raw response dump ─────────────────────────────────────
print('\nSTEP 1: Raw response dump (deva, headword_slp1, match)')
url1 = f'{CSALT_BASE}/mw/restful/entries?field=headword_slp1&query=deva&query_type=match&size=2'
s1, p1, raw1 = http_get_raw(url1)
print(f'  HTTP {s1}')
print(f'  Raw (first 300 chars): {raw1[:300]}')
entries1, shape1 = flatten_entries(p1)
print(f'  Shape detected: {shape1}')
print(f'  Entries found: {len(entries1)}')
if entries1:
    e0 = entries1[0]
    if isinstance(e0, dict):
        print(f'  First entry keys: {list(e0.keys())[:8]}')
        for k in ('headword_slp1', 'headword', 'lemma', 'id'):
            if k in e0: print(f'  {k} = {e0[k]}')
time.sleep(0.5)

# ── STEP 2: GQL schema introspection (null-safe) ─────────────────
print('\nSTEP 2: GQL schema introspection')
# Try root query type first (more reliable than named type)
_, root_resp = gql_post_raw('mw', '{ __schema { queryType { fields { name args { name } } } } }')
if root_resp and isinstance(root_resp, dict):
    data = root_resp.get('data') or {}
    fields = data.get('__schema', {}).get('queryType', {}).get('fields', [])
    if fields:
        print(f'  Root query fields: {[f["name"] for f in fields]}')
        for f in fields:
            args = [a['name'] for a in f.get('args', [])]
            print(f'  {f["name"]}({args})')
    else:
        print(f'  Schema resp (first 300): {str(root_resp)[:300]}')
else:
    print(f'  Schema query failed: {root_resp}')
time.sleep(0.5)

# Try to find the QueryType enum values
_, enum_resp = gql_post_raw('mw', '{ __type(name: "QueryType") { enumValues { name } } }')
if enum_resp and isinstance(enum_resp, dict):
    data2 = enum_resp.get('data') or {}
    qtype_data = data2.get('__type') or {}
    enums = qtype_data.get('enumValues') or []
    if enums:
        names = [e['name'] for e in enums]
        print(f'  QueryType enum values: {names}')
        CONFIG['gql_qtype'] = names[0]
    else:
        print(f'  QueryType introspection: {str(enum_resp)[:300]}')
time.sleep(0.5)

# ── STEP 3: REST field × query_type matrix ────────────────────────
print('\nSTEP 3: REST matrix — field × query_type (word=deva)')
# v5: headword added as first candidate (may be the new correct field)
REST_FIELDS = ['headword', 'headword_slp1', 'lemma', 'form', 'slp1']
REST_QTYPES = ['match', 'term', 'prefix', 'fuzzy', 'match_phrase']
rest_winner = None

for field in REST_FIELDS:
    for qtype in REST_QTYPES:
        url = f'{CSALT_BASE}/mw/restful/entries?field={field}&query=deva&query_type={qtype}&size=1'
        s, parsed, raw = http_get_raw(url, timeout=10)
        entries, shape = flatten_entries(parsed)
        n = len(entries)
        marker = '  ← WORKS' if n > 0 else ''
        print(f'  HTTP {str(s):>3} | {field:<20} | {qtype:<15} | {n} results | shape={shape}{marker}')
        if n > 0 and not rest_winner:
            rest_winner = (field, qtype)
            CONFIG['field'] = field
            CONFIG['qtype'] = qtype
            CONFIG['api_alive'] = True
            if entries[0] and isinstance(entries[0], dict):
                print(f'       First entry keys: {list(entries[0].keys())[:6]}')
        time.sleep(0.3)
    time.sleep(0.3)

# ── STEP 4: GQL field × queryType matrix ─────────────────────────
print('\nSTEP 4: GQL matrix — field × queryType (word=deva)')
GQL_FIELDS = ['headword_slp1', 'headword', 'lemma']
GQL_QTYPES = ['match', 'MATCH', 'term', 'TERM', 'prefix', 'PREFIX']
gql_winner = None

for gfield in GQL_FIELDS:
    for qt in GQL_QTYPES:
        q = f'{{ entries(field: {gfield}, query: "deva", queryType: {qt}, size: 1) {{ {gfield} }} }}'
        gs, gresp = gql_post_raw('mw', q)
        if gresp and isinstance(gresp, dict) and 'data' in gresp:
            data = gresp.get('data') or {}
            gentries = data.get('entries', [])
            n = len(gentries) if gentries else 0
            marker = '  ← WORKS' if n > 0 else ''
            print(f'  HTTP {gs} | field:{gfield:<20} queryType:{qt:<8} → {n} results{marker}')
            if n > 0 and not gql_winner:
                gql_winner = (gfield, qt)
                CONFIG['gql_field'] = gfield
                CONFIG['gql_qtype'] = qt
                CONFIG['api_alive'] = True
        else:
            err = str(gresp)[:80] if gresp else 'None'
            print(f'  HTTP {gs} | field:{gfield:<20} queryType:{qt:<8} → error: {err}')
        time.sleep(0.5)
    time.sleep(0.3)

# ── VERDICT ──────────────────────────────────────────────────────
print('\n' + '=' * 65)
print('DIAGNOSTIC VERDICT')
print('=' * 65)
if rest_winner:
    CONFIG['use_gql'] = False
    print(f'✅ REST: field={rest_winner[0]}, query_type={rest_winner[1]}')
elif gql_winner:
    CONFIG['use_gql'] = True
    print(f'✅ GQL: field={gql_winner[0]}, queryType={gql_winner[1]}')
else:
    print('❌ All REST and GQL combinations returning 0 or errors.')
    print('   The API may be temporarily down or undergoing maintenance.')
    print('   → Wait 30–60 min, restart Colab runtime, rerun Cell 2.')
    print('   → Manual check: https://api.c-salt.uni-koeln.de/dicts/mw/restful')
    print('   → Contact: info-csalt@uni-koeln.de')

print(f'\nCONFIG = {CONFIG}')
print('→ Proceed to Cell 3 only if api_alive=True')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 3 — SMART LOOKUP FUNCTIONS + PING
# ════════════════════════════════════════════════════════════════
if not CONFIG['api_alive']:
    print('⚠ API not alive — rerun Cell 2 or wait and restart.')
else:
    def slp1_variants(word):
        v = [word]
        if word.lower() != word: v.append(word.lower())
        return v

    def rest_lookup(word, dict_id, size=3):
        for variant in slp1_variants(word):
            try:
                params = urllib.parse.urlencode({
                    'field': CONFIG['field'], 'query': variant,
                    'query_type': CONFIG['qtype'], 'size': str(size)
                })
                url = f'{CSALT_BASE}/{dict_id}/restful/entries?{params}'
                req = urllib.request.Request(url, headers=H_REST)
                with urllib.request.urlopen(req, timeout=20) as r:
                    raw = json.loads(r.read())
                entries, _ = flatten_entries(raw)
                if entries: return True, entries, f'REST({variant})'
            except urllib.error.HTTPError as e:
                if e.code == 429:
                    STATE['total_429'] += 1
                    wait = min(30 * (2 ** (STATE['total_429']-1)), 120)
                    print(f'  ⚠ 429! Wait {wait}s...'); time.sleep(wait)
                elif e.code == 403:
                    STATE['total_403'] += 1
                    if STATE['total_403'] >= 3: CONFIG['use_gql'] = True; return None,[],'REST_403→GQL'
                    time.sleep(30)
                elif e.code != 404:
                    return None, [], f'REST_HTTP_{e.code}'
            except OSError as e:
                if 'unreachable' in str(e).lower(): STATE['network_dead']=True; return None,[],'NETWORK_DEAD'
                return None, [], 'REST_OS_ERROR'
        return False, [], 'NOT_FOUND'

    def gql_lookup(word, dict_id, size=3):
        gfield = CONFIG.get('gql_field') or 'headword_slp1'
        qt     = CONFIG.get('gql_qtype') or 'match'
        for variant in slp1_variants(word):
            try:
                q = f'{{ entries(field: {gfield}, query: "{variant}", queryType: {qt}, size: {size}) {{ {gfield} body }} }}'
                payload = json.dumps({'query': q}).encode()
                req = urllib.request.Request(GQL_URL[dict_id], data=payload, headers=H_GQL)
                with urllib.request.urlopen(req, timeout=30) as r:
                    result = json.loads(r.read())
                entries = (result.get('data') or {}).get('entries', [])
                if entries: return True, entries, f'GQL({variant},{qt})'
            except urllib.error.HTTPError as e:
                if e.code == 429: print('  ⚠ GQL 429!'); time.sleep(60)
                elif e.code == 400: CONFIG['gql_qtype'] = 'match' if qt=='MATCH' else 'MATCH'
                else: return None, [], f'GQL_HTTP_{e.code}'
            except OSError as e:
                if 'unreachable' in str(e).lower(): STATE['network_dead']=True; return None,[],'NETWORK_DEAD'
        return False, [], 'GQL_NOT_FOUND'

    def smart_lookup(word, dict_id, size=3):
        if STATE['network_dead']: return None, [], 'SKIP'
        STATE['calls'] += 1
        if not CONFIG['use_gql']:
            found, entries, ep = rest_lookup(word, dict_id, size)
            if found is True:  STATE['consec_errors']=0; return True, entries, ep
            if found is False: STATE['consec_errors']=0; return False, [], ep
            STATE['consec_errors'] += 1
            if STATE['network_dead']: return None, [], ep
            if STATE['consec_errors'] >= 5 or 'GQL' in ep: CONFIG['use_gql']=True
            else: return None, [], ep
        time.sleep(1.0)
        found, entries, ep = gql_lookup(word, dict_id, size)
        if found is True: STATE['consec_errors']=0
        elif found is None: STATE['consec_errors']+=1
        return found, entries, ep

    def safe_hw(entry):
        if not isinstance(entry, dict): return str(entry)[:30]
        return (entry.get('headword_slp1') or entry.get('headword') or
                entry.get('lemma') or entry.get('id') or '?')

    mode = f'GQL({CONFIG["gql_field"]},{CONFIG["gql_qtype"]})' if CONFIG['use_gql'] \
           else f'REST(field={CONFIG["field"]},qtype={CONFIG["qtype"]})'
    print(f'Using: {mode}')
    print('Ping: deva, agni, rudra × 3 dicts')
    for d in DICTS:
        hits = 0
        for w in ['deva','agni','rudra']:
            f,e,ep = smart_lookup(w,d,1)
            if f is True: hits+=1; hw=safe_hw(e[0]) if e else '?'; print(f'  ✓[{d}]{w}→{hw}')
            else: print(f'  ✗[{d}]{w}({ep})')
            if STATE['network_dead']: break
            time.sleep(CONFIG['delay'])
        if STATE['network_dead']: break
        time.sleep(0.5)
    print('\n→ Proceed to Cell 4.')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 4 — MAIN SCAN  (24 words × 3 dicts)
# ════════════════════════════════════════════════════════════════
if not CONFIG['api_alive']:
    print('⚠ Skipping.')
else:
    print(f'MAIN SCAN — {len(WORDS)} words × 3 dicts')
    print('=' * 65)
    RESULTS = {}

    for i, (slp1, human, epithet, claim) in enumerate(WORDS):
        if STATE['network_dead']: print(f'Network dead at word {i+1}.'); break
        RESULTS[slp1] = {'human':human,'epithet':epithet,'claim':claim,'dicts':{}}
        row = f'  {i+1:>2}/{len(WORDS)} {human:<12}'
        for dict_id in DICTS:
            if STATE['network_dead']: break
            found, entries, ep = smart_lookup(slp1, dict_id, size=3)
            hw_list = [safe_hw(e) for e in entries[:2]] if found else []
            RESULTS[slp1]['dicts'][dict_id] = {
                'found': bool(found) if found is not None else None,
                'endpoint': ep, 'headwords': hw_list
            }
            if found is True:  row += f'  ✓{dict_id}:{hw_list[0] if hw_list else "?"}'
            elif found is False: row += f'  ✗{dict_id}'
            else: row += f'  ?{dict_id}'
            time.sleep(1.0 if CONFIG['use_gql'] else CONFIG['delay'])
        print(row)
        time.sleep(0.5)

    sc=len(RESULTS)
    af=sum(1 for r in RESULTS.values() if any(r['dicts'].get(d,{}).get('found') for d in DICTS))
    a3=sum(1 for r in RESULTS.values() if all(r['dicts'].get(d,{}).get('found') for d in DICTS))
    print(f'\nScanned:{sc}/{len(WORDS)}  AnyDict:{af}  All3:{a3}')
    print(f'Calls:{STATE["calls"]}  429s:{STATE["total_429"]}  403s:{STATE["total_403"]}')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 5 — DEEP CHECKS
# ════════════════════════════════════════════════════════════════
deep = {}
if not CONFIG['api_alive'] or STATE['network_dead']:
    print('Skipping.')
else:
    print('DEEP CHECKS'); print('=' * 55)

    def body_check(slp1, dict_id, terms, size=5):
        found, entries, ep = smart_lookup(slp1, dict_id, size=size)
        time.sleep(1.0 if CONFIG['use_gql'] else CONFIG['delay'])
        if not found: return {'found':False,'matched':[],'snippet':'','ep':ep}
        body = ' '.join((e.get('body','') or '') + ' ' for e in entries if isinstance(e,dict)).lower()
        matched = [t for t in terms if t.lower() in body]
        return {'found':True,'matched':matched,'snippet':body[:300],'ep':ep}

    for qnum, slp1, label, terms in [
        ('Q1','ambaka','ambaka: eye AND/OR mother?',['eye','mother','amba','maternal']),
        ('Q2','sadA','sadā: adverb "always"?',['always','ever','ind','sat']),
        ('Q3','mahat','mahat: Sāṃkhya?',['sankhya','samkhya','intellect','buddhi','cosmic']),
        ('Q4','yuj','√yuj: to yoke?',['yoke','join','unite','yoga','harness']),
        ('Q5','kAla','kāla: time AND death?',['time','death','yama','black','end']),
    ]:
        print(f'\n{qnum}: {label}')
        r = body_check(slp1,'mw',terms)
        deep[qnum] = r
        print(f'  Found:{r["found"]}  Matched:{r["matched"]}')
        if r['snippet']: print(f'  Snippet: {r["snippet"][:200]}')

    print(f'\nTotal calls: {STATE["calls"]}')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 6 — PAPER OUTPUT + SAVE
# ════════════════════════════════════════════════════════════════
print('PAPER OUTPUT'); print('=' * 65)

EP_NAMES = {1:'Viśveśvarāya',2:'Mahādevāya',3:'Tryambakāya',4:'Tripurāntakāya',
            5:'Trikālagnikālāya',6:'Kālāgnirudrāya',7:'Nīlakaṇṭhāya',8:'Mṛtyuñjayāya',
            9:'Sarveśvarāya',10:'Sadāśivāya',11:'Śrīmanmahādevāya'}

if 'RESULTS' not in dir() or not RESULTS:
    print('No results. Run Cells 1–4 first.')
else:
    edata = defaultdict(lambda:{'words':[],'scanned':0,'all_found':True,'any_found':False})
    for slp1,r in RESULTS.items():
        ep=r['epithet']
        any_f = any(r['dicts'].get(d,{}).get('found') for d in DICTS)
        all_f = all(r['dicts'].get(d,{}).get('found') for d in DICTS)
        mw_f  = r['dicts'].get('mw',{}).get('found',False)
        edata[ep]['words'].append({'word':r['human'],'mw':mw_f,'any':any_f})
        edata[ep]['scanned'] += 1
        if any_f: edata[ep]['any_found']=True
        if not all_f: edata[ep]['all_found']=False

    confirmed=partial=unconfirmed=not_scanned=0
    print(f'\n{"#":<3}{"Epithet":<26}{"Words":<32}{"Status":>12}')
    print('-'*77)
    for ep_num in range(1,12):
        ep=EP_NAMES[ep_num]; d=edata[ep_num]
        ws=', '.join(f'{w["word"]}({"✓" if w["any"] else "✗"})' for w in d['words'])
        if not ws: ws='(not reached)'
        if d['scanned']==0:    verdict='NOT SCANNED'; not_scanned+=1
        elif d['all_found']:   verdict='CONFIRMED';   confirmed+=1
        elif d['any_found']:   verdict='PARTIAL';     partial+=1
        else:                  verdict='UNCONFIRMED'; unconfirmed+=1
        print(f'{ep_num:<3}{ep:<26}{ws:<32}{verdict:>12}')
    print('-'*77)
    sc=confirmed+partial+unconfirmed
    print(f'  Confirmed:{confirmed}  Partial:{partial}  Unconfirmed:{unconfirmed}  NotScanned:{not_scanned}')

    run_date=datetime.now().strftime('%d %B %Y')
    if not_scanned==0 and unconfirmed==0:
        note=(f"C-SALT verification completed {run_date}: all component words of the 11 Śivopāsanā "
              f"epithets confirmed in MW [14], Apte (ap90), and Böhtlingk-Roth (pwg) "
              f"({confirmed}/11 confirmed in all three; {partial}/11 partial). "
              f"Key: ambaka attests both eye and maternal readings; sadā confirmed as adverb; "
              f"mahat confirmed with Sāṃkhya usage; kāla confirmed with both time and death. Notebook [24].")
    elif not_scanned>0:
        note=(f"C-SALT partial verification {run_date}: {sc}/11 epithets scanned "
              f"({confirmed} confirmed; {partial} partial; {not_scanned} not reached). "
              f"Notebook [24]: github.com/ZPRM/panchakshari-vedic-timecycles")
    else:
        note=(f"C-SALT verification {run_date}: {confirmed}/11 confirmed; {partial}/11 partial; "
              f"{unconfirmed}/11 unconfirmed. Notebook [24].")

    print(f'\nPAPER NOTE:\n  "{note}"')

    ts=datetime.now().strftime('%Y%m%d_%H%M')
    report={
        'version':'v5','date':run_date,'paper':'Vedic Cosmological Time Cycles v8.6',
        'config':CONFIG,'state':STATE,
        'words_scanned':len(RESULTS),'words_total':len(WORDS),
        'results':{s:{'human':r['human'],'epithet':r['epithet'],
                      'dicts':{k:{'found':v.get('found'),'ep':v.get('endpoint'),
                                  'headwords':v.get('headwords',[])} for k,v in r['dicts'].items()}}
                   for s,r in RESULTS.items()},
        'deep':{q:{'found':v.get('found'),'matched':v.get('matched',[])} for q,v in deep.items()},
        'summary':{'confirmed':confirmed,'partial':partial,'unconfirmed':unconfirmed,
                   'not_scanned':not_scanned,'calls':STATE['calls']},
        'paper_note':note,
    }
    out=f'{OUTPUT_DIR}/csalt_paper_verification_{ts}.json'
    with open(out,'w',encoding='utf-8') as f: json.dump(report,f,ensure_ascii=False,indent=2)
    print(f'\n✅ Saved: {out}')
    print('Upload JSON to Claude → final paper update.')